In [1]:
# Ô 1 – Kết nối Drive, kiểm tra GPU
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/EdgeAI'

GPU 0: Tesla T4 (UUID: GPU-dbd39849-3a9f-2337-e443-4ba292a9ddeb)
Mounted at /content/drive


In [2]:
# Ô 2 – Giải nén code + dữ liệu intent MỚI (luôn ghi đè), chạy test nhanh
%cd /content
!rm -rf /content/TextCNN && mkdir -p /content/TextCNN && unzip -q -o "$DRIVE/textcnn_code.zip" -d /content/TextCNN
%cd /content/TextCNN
!PYTHONPATH=src python -m pytest tests/test_textcnn.py -q -p no:cacheprovider 2>&1 | tail -1
!cat configs/textcnn_queue.json

/content
/content/TextCNN
5 passed, 6 warnings in 12.77s
{
  "_note": "Thứ tự thí nghiệm TextCNN mà scripts/colab_intent.py chạy. Mỗi tên ứng với configs/<tên>.json. 'use': tên model intent muốn nộp (null = tự chọn theo stress INT8).",
  "queue": ["textcnn_word", "textcnn_char", "textcnn_dual"],
  "use": null
}


In [ ]:
# Ô 3 – Chạy hàng đợi: CV -> stress -> INT8 -> kiểm tra luật -> submission, rồi in bảng + ghép submission
!python scripts/colab_intent.py --drive "$DRIVE"

Hàng đợi: ['textcnn_word', 'textcnn_char', 'textcnn_dual']

========== textcnn_word
GPU: ['/physical_device:GPU:0']   TF 2.21.0
config: {'branches': {'word': {'E': 64, 'nf': 96, 'ks': [1, 2, 3, 4]}}, 'epochs': 25, 'bs': 64, 'lr': 0.002}
== cv ==
  CV fold 1/5: FP32 1.0000  INT8 1.0000
  CV fold 2/5: FP32 1.0000  INT8 1.0000
  CV fold 3/5: FP32 1.0000  INT8 1.0000
  CV fold 4/5: FP32 1.0000  INT8 1.0000
  CV fold 5/5: FP32 1.0000  INT8 1.0000
== stress ==
  stress 'vi tri' (n=227): FP32 0.2731  INT8 0.2863
  stress 'phia' (n=184): FP32 0.6902  INT8 0.6902
  stress 'mep' (n=46): FP32 1.0000  INT8 1.0000
  stress 'ngang' (n=159): FP32 1.0000  INT8 1.0000
  stress 'xac dinh' (n=100): FP32 1.0000  INT8 1.0000
  stress 'dem' (n=679): FP32 0.8733  INT8 0.8778
  stress 'so luong' (n=107): FP32 1.0000  INT8 1.0000
  stress 'bao nhieu cai' (n=134): FP32 0.9254  INT8 0.9254
  stress 'tat ca' (n=282): FP32 1.0000  INT8 1.0000
  stress 'xuat hien' (n=524): FP32 1.0000  INT8 1.0000
  stress 'nhin th

In [ ]:
# Ô 4 – Xem kỹ từng thí nghiệm: đường cong train, stress theo từng cụm từ, câu đoán sai
import json, os
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
for name in json.load(open('configs/textcnn_queue.json'))['queue']:
    d = f'{DRIVE}/experiments_intent/{name}'
    if not os.path.exists(f'{d}/report.json'):
        continue
    rep = json.load(open(f'{d}/report.json', encoding='utf-8'))
    log, st = pd.read_csv(f'{d}/log.csv'), pd.read_csv(f'{d}/stress.csv')
    fig, ax = plt.subplots(1, 3, figsize=(16, 4)); fig.suptitle(f"{name}  ({rep['params']:,} tham số, {rep['file_bytes']//1024} KB INT8)")
    log.plot(x='epoch', y='loss', ax=ax[0], title='loss (model cuối)')
    log.plot(x='epoch', y='accuracy', ax=ax[1], title='accuracy train (model cuối)')
    st.set_index('cue')[['acc_fp32', 'acc_int8']].plot.barh(ax=ax[2], xlim=(0, 1), title='stress: acc khi giấu cụm từ')
    plt.tight_layout(); plt.show()
    print('Kiểm tra luật:', rep['checks'], '| op:', rep['ops'])
    print('Test (không có nhãn, KHÔNG phải accuracy):', rep['test (không có nhãn -> KHÔNG phải accuracy)'])
    print('Câu đoán sai trong CV:'); display(pd.read_csv(f'{d}/cv_errors.csv').head(10))